# Séance 11 — Surveillance : dérive des données et de la performance
*Session 11 — Monitoring: data and performance drift*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/11_surveillance_derive.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ○ démonstration |

**À la fin de cette séance, vous saurez :**

1. calculer l'indice de stabilité de population (PSI) et l'interpréter ;
2. compléter le PSI par le test de Kolmogorov-Smirnov ;
3. distinguer dérive des données, dérive opérationnelle et dérive de performance ;
4. tenir compte de la maturité des étiquettes pour mesurer le rappel.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

In [ ]:
cfg = preparer("surveiller")
s = cfg["surveillance"]
dossier = cfg.chemin("artefacts") / "surveillance"
rap = lire_json(dossier / "rapport_surveillance.json")
print("Réentraînement recommandé :", rap.get("reentrainement_recommande"))
pd.DataFrame(rap["alertes"])

In [ ]:
for f in ["20_derive_psi.png", "21_charge_alertes.png", "22_performance.png"]:
    image(dossier / f)

$$\mathrm{PSI} = \sum_{k} (p^{cour}_k - p^{ref}_k)\,\ln\frac{p^{cour}_k}{p^{ref}_k}$$
où les classes $k$ sont les déciles de la distribution de **référence**. Repères usuels : < 0,10 stable; 0,10–0,25 à surveiller;
> 0,25 dérive importante (seuils `psi_avertissement` et `psi_alerte` de la configuration).

### Exercice 11.1 — 🟢 Tous les parcours

Écrivez `psi(reference, courant, n_classes=10)` : retirez les valeurs non finies; bornes = quantiles de la référence
(`np.quantile(reference, np.linspace(0, 1, n_classes + 1))`, puis `np.unique`), première et dernière bornes remplacées par −∞ et +∞;
proportions par classe avec `np.histogram`, bornées inférieurement à 1e-6; puis la formule.

In [ ]:
# VOTRE CODE ICI
def psi(reference, courant, n_classes=10):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
from fraude360.monitoring import surveillance
_r = np.random.default_rng(0).lognormal(3, 1, 5000)
_c = np.random.default_rng(1).lognormal(3.3, 1, 800)
verifier_proche(psi(_r, _c), surveillance.psi(_r, _c), 1e-9)
verifier(psi(_r, _r) < 1e-9, "Une distribution comparée à elle-même : PSI nul")

In [ ]:
gold = table("gold", "caracteristiques")
gold["horodatage"] = pd.to_datetime(gold["horodatage"])
gold["jour_index"] = (gold["horodatage"] - gold["horodatage"].min().normalize()).dt.days
d = cfg["decoupage"]
reference = gold[gold.jour_index < d["jours_entrainement"]]
production = gold[gold.jour_index >= d["jours_entrainement"] + d["jours_validation"]]
print(len(reference), "transactions de référence,", len(production), "en production")

### Exercice 11.2 — 🟢 Tous les parcours

Calculez `psi_jour`, un `DataFrame` avec une ligne par **jour** de production et une colonne par variable
pour `montant` et `secondes_depuis_tx_precedente` (ignorez les journées incomplètes de moins de 20 transactions). Ajoutez une colonne `niveau_secondes` : « stable », « avertissement » ou « alerte »
selon les seuils de `s`. Tracez les deux séries.

In [ ]:
# VOTRE CODE ICI
psi_jour = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_der = pd.read_csv(dossier / "derive_donnees.csv")
_m = _der[_der.variable == "montant"].set_index("date")["psi"]
_m.index = pd.to_datetime(_m.index).date
_comm = [j for j in psi_jour.index if j in _m.index]
verifier(len(_comm) > 0 and np.allclose(psi_jour.loc[_comm, "montant"].round(4), _m.loc[_comm], atol=2e-4),
         "Vos PSI de montant doivent égaler ceux du pipeline")

### Exercice 11.3 — 🔷 Informatique — approfondissement

Pour le **dernier** jour de production, calculez la statistique KS et sa valeur p (`scipy.stats.ks_2samp`) pour
`secondes_depuis_tx_precedente`. Stockez-les dans `ks_stat, ks_p`. Pourquoi une valeur p minuscule ne suffit-elle pas à déclencher
une alerte avec des milliers d'observations ?

In [ ]:
# VOTRE CODE ICI
from scipy import stats
ks_stat, ks_p = ..., ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(0 <= ks_stat <= 1 and 0 <= ks_p <= 1)

**❓ Question (🔶 Gestion — décision d'affaires) —** Le PSI d'une variable est en alerte, mais le rappel mesuré sur les étiquettes mûres est stable. Réentraînez-vous ? Que vérifiez-vous d'abord ?

*Votre réponse ici.*

## Bilan

- PSI et KS détectent un changement de distribution, pas une baisse de performance.
- La performance ne se mesure que sur des étiquettes mûres (délai de contestation).
- La charge d'alertes est un indicateur opérationnel aussi important que les métriques du modèle.

**Pour aller plus loin :**

- `src/fraude360/monitoring/surveillance.py` et les tables Gold `surveillance_*` (Power BI).

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*